# 🥈 Camada Prata — Limpeza, Padronização, Engenharia de Atributos e Análise

*Segunda etapa do pipeline (arquitetura Bronze → Prata): lê as tabelas brutas, trata, enriquece e gera os datasets finais.*

---

## 1. Introdução

### 🎯 Objetivo
Este notebook é a **Camada Prata** do projeto de Análise de Ações da B3. Ele lê as tabelas brutas geradas pela **Camada Bronze** (`bronze.cotacoes_b3` e `bronze.cadastro_empresas_b3`) e aplica:

1. **Limpeza** — remoção de inconsistências (cotações ausentes, duplicatas, cadastros inválidos);
2. **Padronização** — tradução de setor/indústria para português e tipos de dados consistentes;
3. **Engenharia de atributos** — variação diária, amplitude, retorno diário, média móvel de 20 dias;

O resultado é persistido como a tabela curada `prata.acoes_b3`, a partir da qual geramos a **análise exploratória** e os **datasets finais** (CSV para Power BI e para Analytics).

> ⚠️ **Pré-requisito:** este notebook depende das tabelas criadas pelo `Finance_project_Bronze.ipynb`. Execute-o primeiro.

### 🛠️ Tecnologias Utilizadas
| Tecnologia | Papel nesta camada |
|---|---|
| **PySpark** | Limpeza, padronização e engenharia de atributos |
| **Delta Lake** | Leitura da Bronze e persistência da tabela curada (Prata) |
| **Pandas** | Tradução de setor/indústria e exportação dos CSVs finais |

---

## 2. Importação das Bibliotecas

In [0]:
import os
import sys

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 3. Configuração do Ambiente

Inicializamos o Spark e criamos (se ainda não existir) o schema `prata`, onde a tabela curada desta camada será salva.

In [0]:
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = SparkSession.builder.appName("Prata_Acoes_B3").getOrCreate()

# Schema da camada Prata
# 💡 Se você já usa outro nome de schema/catalog no seu workspace, basta ajustar aqui
spark.sql("CREATE DATABASE IF NOT EXISTS prata")

DataFrame[]

## 4. Leitura da Camada Bronze

Lemos as duas tabelas geradas pelo notebook anterior. O cadastro setorial é convertido para pandas, pois a tradução de setor/indústria (Seção 5.2) é feita com `.map()`, mais simples e legível do que uma longa cadeia de `.when()` em Spark.

In [0]:
df_dados = spark.read.table("workspace.bronze.table_ing_b3")
df_setores_pd = spark.read.table("workspace.bronze.table_cadastro_empresas").toPandas()

print(f"Linhas lidas de bronze.cotacoes_b3: {df_dados.count()}")
print(f"Linhas lidas de bronze.cadastro_empresas_b3: {len(df_setores_pd)}")

Linhas lidas de bronze.cotacoes_b3: 132902
Linhas lidas de bronze.cadastro_empresas_b3: 56


## 5. ETL — Limpeza, Padronização e Engenharia de Atributos

### 5.1 Limpeza

Removemos:
- linhas sem cotação válida (datas em que a ação não foi negociada);
- registros duplicados (mesma data + mesmo ticker);
- tickers sem informação cadastral válida (falhas de consulta ao `yfinance`).

In [0]:
linhas_antes = df_dados.count()

df_dados = (
    df_dados
    .dropna(subset=["Abertura", "Minima", "Maxima", "Fechamento"])   # remove dias sem cotação
    .dropDuplicates(["data", "Ticker"])                 # remove duplicidade de data + ticker
)

linhas_depois = df_dados.count()
print(f"Linhas removidas na limpeza de cotações: {linhas_antes - linhas_depois}")

empresas_antes = len(df_setores_pd)
df_setores_pd = df_setores_pd.dropna(subset=["Setor", "Nome_Empresa"]).drop_duplicates(subset=["Codigo"])
print(f"Tickers removidos na limpeza de cadastro: {empresas_antes - len(df_setores_pd)}")

Linhas removidas na limpeza de cotações: 0
Tickers removidos na limpeza de cadastro: 0


### 5.2 Padronização

O `yfinance` retorna setor e indústria em inglês. Para facilitar a leitura no Power BI e em relatórios em português, criamos dois dicionários de tradução (`Setor` → `setor_atuacao` e `Industria` → `segmento`) e os aplicamos diretamente no pandas — uma forma mais legível e fácil de manter do que uma longa cadeia de `.when()`.

Em seguida, unimos cotações e cadastro, e padronizamos nomes de colunas (em português) e tipos numéricos.

> 💡 Mantemos os preços como `double` (numérico) durante todo o pipeline. A formatação no padrão brasileiro (vírgula decimal) só é aplicada na exportação do CSV para Power BI, na Seção 8.1 — isso evita perda de precisão nos cálculos da próxima etapa.

In [0]:
setor_traducao = {
    "Financial Services": "Financeiro",
    "Health Care": "Saude",
    "Consumer Defensive": "Consumo Defensivo",
    "Consumer Cyclical": "Consumo Cíclico",
    "Real Estate": "Imobiliario",
    "Basic Materials": "Materiais Basicos",
    "Energy": "Energia",
    "Industrials": "Industrial",
    "Consumer Discretionary": "Consumo Discricionario",
    "Consumer Staples": "Consumo Essencial",
    "Utilities": "Servicos Publicos",
    "Communication Services": "Servicos de Comunicacao",
    "Healthcare": "Saude",
    "Technology": "Tecnologia",
    "Fixtures & Accessories": "Acessorios e Utensilios",
    "Consumer Services": "Servicos de Consumo",
}

In [0]:
segmento_traducao = {
    "Financial Data & Stock Exchanges": "Dados Financeiros e Bolsas de Valores",
    "Real Estate - Development": "Imobiliario - Incorporacao",
    "Other Industrial Metals & Mining": "Outros Metais Industriais e Mineracao",
    "Banks - Regional": "Bancos Regionais",
    "Insurance - Life": "Seguros de Vida",
    "Oil & Gas Integrated": "Petroleo e Gas - Integrado",
    "Airlines": "Companhias Aereas",
    "Steel": "Aco",
    "Education & Training Services": "Educacao e Servicos de Treinamento",
    "Department Stores": "Lojas de Departamento",
    "Beverages - Brewers": "Bebidas - Cervejarias",
    "Grocery Stores": "Supermercados",
    "Railroads": "Ferrovias",
    "Conglomerates": "Conglomerados",
    "Oil & Gas E&P": "Exploracao e Producao de Petroleo e Gas",
    "Utilities - Renewable": "Utilidades - Energias Renovaveis",
    "Specialty Retail": "Varejo Especializado",
    "Specialty Business Services": "Servicos Empresariais Especializados",
    "Utilities - Diversified": "Utilidades - Diversificadas",
    "Farm & Heavy Construction Machinery": "Maquinas Agricolas e de Construcao Pesada",
    "Travel Services": "Servicos de Viagem",
    "Oil & Gas Refining & Marketing": "Refino e Comercializacao de Petroleo e Gas",
    "Marine Shipping": "Transporte Maritimo",
    "Aluminum": "Aluminio",
    "Utilities - Regulated Electric": "Utilidades - Eletricidade Regulada",
    "Electrical Equipment & Parts": "Equipamentos e Pecas Eletricas",
    "Rental & Leasing Services": "Servicos de Aluguel e Arrendamento",
    "Packaged Foods": "Alimentos Embalados",
    "Capital Markets": "Mercados de Capitais",
    "Insurance - Diversified": "Seguros - Diversificados",
    "Residential Construction": "Construcao Residencial",
    "Telecom Services": "Servicos de Telecomunicacoes",
    "Healthcare Plans": "Planos de Saude",
    "Farm Products": "Produtos Agricolas",
    "Paper & Paper Products": "Papel e Produtos de Papel",
    "Asset Management": "Gestao de Ativos",
    "Infrastructure Operations": "Operacoes de Infraestrutura",
    "Household & Personal Products": "Produtos Domesticos e Pessoais",
    "Medical Care Facilities": "Instalacoes de Cuidados Medicos",
    "Aerospace & Defense": "Aeroespacial e Defesa",
    "Lumber & Wood Production": "Producao de Madeira e Serraria",
    "Pharmaceutical Retailers": "Farmecias e Varejo Farmaceutico",
    "Software - Infrastructure": "Software - Infraestrutura",
    "Real Estate Services": "Servicos Imobiliarios",
    "Electronics & Computer Distribution": "Distribuição de Eletronicos e Computadores",
    "Software - Application": "Software - Aplicacoes",
    "Utilities - Regulated Water": "Utilidades - Agua Regulada",
    "Chemicals": "Produtos Quimicos",
    "Footwear & Accessories": "Calcados e Acessorios",
    "Diagnostics & Research": "Diagnostico e Pesquisa",
    "Luxury Goods": "Bens de Luxo",
    "Internet Retail": "Varejo Online",
    "Medical Distribution ": "Distribuicao Medica",
    "Communication Equipment": "Equipamentos de Comunicacao",
    "Drug Manufacturers - Specialty & Generic": "Fabricantes de Medicamentos - Especializados e Genéricos",
    "Integrated Freight & Logistics": "Transporte e Logistica Integrada",
    "Engineering & Construction": "Engenharia e Construcao",
    "Specialty Chemicals": "Produtos Químicos Especiais",
    "Internet Content & Information": "Conteudo e Informacao na Internet",
    "Leisure": "Lazer",
    "Computer Hardware": "Hardware de Computadores",
    "Specialty Industrial Machinery": "Maquinas Industriais Especiais",
    "Waste Management": "Gerenciamento de Residuos",
    "Auto Parts": "Pecas Automotivas",
    "Confectioners": "Confeitaria",
    "Restaurants": "Restaurantes",
    "Packaging & Containers ": "Embalagens e Recipientes",
    "Insurance - Reinsurance": "Resseguros",
    "Agricultural Inputs": "Insumos Agricolas",
    "Apparel Retail": "Varejo de Roupas",
    "Building Products & Equipment ": "Produtos e Equipamentos para Construcao",
    "Real Estate - Diversified": "Imobiliario - Diversificado",
    "Metal Fabrication": "Fabricacao de Metais",
    "Drug Manufacturers - General": "Fabricantes de Medicamentos - Gerais",
    "Insurance Brokers": "Corretores de Seguros",
    "Entertainment": "Entretenimento",
    "Industrial Distribution": "Distribuicao Industrial",
    "Copper": "Cobre",
    "Trucking": "Transporte Rodoviario",
    "Personal Services": "Servicos Pessoais",
    "Advertising Agencies": "Agencias de Publicidade",
    "Textile Manufacturing": "Industria Textil",
    "Oil & Gas Equipment & Services": "Equipamentos e Servicos para Petroleo e Gas",
    "Biotechnology": "Biotecnologia",
    "Furnishings": "Mobiliario e Decoracao",
    "Apparel Manufacturing": "Industria de Vestuario",
    "Utilities - Regulated Gas": "Utilidades - Gas Regulamentado",
    "Auto & Truck Dealerships": "Concessionarias de Automoveis e Caminhoes",
    "Lodging": "Hospedagem",
    "Medical Instruments & Supplies": "Instrumentos e Suprimentos Medicos",
    "Shell Companies ": "Empresas de Proposito Especifico (Shell Companies)",
}

In [0]:
setor_traducao_expr = F.create_map([F.lit(x) for x in sum(setor_traducao.items(), ())])
segmento_traducao_expr = F.create_map([F.lit(x) for x in sum(segmento_traducao.items(), ())])

df_setores = (
    spark.createDataFrame(df_setores_pd)
    .withColumn("setor_atuacao", F.coalesce(setor_traducao_expr.getItem(F.col("Setor")), F.col("Setor")))
    .withColumn("segmento", F.coalesce(segmento_traducao_expr.getItem(F.col("Industria")), F.col("Industria")))
)

df_setores = spark.createDataFrame(df_setores_pd)
display(df_setores.limit(10))

Codigo,Setor,Industria,Nome_Empresa,setor_atuacao,segmento
BBDC4.SA,Financial Services,Banks - Regional,Banco Bradesco S.A.,Financeiro,Bancos Regionais
BBDC3.SA,Financial Services,Banks - Regional,Banco Bradesco S.A.,Financeiro,Bancos Regionais
ITUB4.SA,Financial Services,Banks - Regional,Itaú Unibanco Holding S.A.,Financeiro,Bancos Regionais
ITUB3.SA,Financial Services,Banks - Regional,Itaú Unibanco Holding S.A.,Financeiro,Bancos Regionais
ITSA4.SA,Industrials,Conglomerates,Itaúsa S.A.,Industrial,Conglomerados
ITSA3.SA,Industrials,Conglomerates,Itaúsa S.A.,Industrial,Conglomerados
BBAS3.SA,Financial Services,Banks - Regional,Banco do Brasil S.A.,Financeiro,Bancos Regionais
SANB11.SA,Financial Services,Banks - Regional,Banco Santander (Brasil) S.A.,Financeiro,Bancos Regionais
SANB4.SA,Financial Services,Banks - Regional,Banco Santander (Brasil) S.A.,Financeiro,Bancos Regionais
SANB3.SA,Financial Services,Banks - Regional,Banco Santander (Brasil) S.A.,Financeiro,Bancos Regionais


In [0]:
df_acoes = df_dados.alias("dados").join(
    df_setores.alias("setor"),
    F.col("dados.Ticker") == F.col("setor.Codigo"),
    "inner"
)

tb_acoes = df_acoes.select(
    F.col("dados.data"),
    F.col("setor.Nome_Empresa").alias("empresa"),
    F.col("setor.setor_atuacao").alias("setor"),
    F.col("setor.segmento").alias("segmento"),
    F.col("dados.Ticker").alias("cd_empresa"),
    F.col("dados.Abertura"),
    F.col("dados.Maxima"),
    F.col("dados.Minima"),
    F.col("dados.Fechamento"),
    F.col("dados.Vol")
)

display(tb_acoes.limit(10))

data,empresa,setor,segmento,cd_empresa,Abertura,Maxima,Minima,Fechamento,Vol
2019-02-25T00:00:00.000Z,Itaúsa S.A.,Industrial,Conglomerados,ITSA3.SA,7.173580,7.336510,7.080477,7.308580,128570
2019-02-25T00:00:00.000Z,TOTVS S.A.,Tecnologia,Software - Aplicacoes,TOTS3.SA,10.400524,10.533553,10.176792,10.176792,1549027
2019-02-26T00:00:00.000Z,Ambev S.A.,Consumo Defensivo,Bebidas - Cervejarias,ABEV3.SA,13.466137,13.531683,13.327761,13.415157,12685600
2019-02-26T00:00:00.000Z,Equatorial S.A.,Servicos Publicos,Utilidades - Eletricidade Regulada,EQTL3.SA,14.078271,14.169196,14.010920,14.169196,3025575
2019-02-26T00:00:00.000Z,Lojas Renner S.A.,Consumo Cíclico,Lojas de Departamento,LREN3.SA,26.131062,26.284079,25.907417,25.960384,2239421
2019-02-26T00:00:00.000Z,Petróleo Brasileiro S.A. - Petrobras,Energia,Petroleo e Gas - Integrado,PETR4.SA,7.164263,7.315203,7.161568,7.212780,46504200
2019-02-26T00:00:00.000Z,Usinas Siderúrgicas de Minas Gerais S.A.,Materiais Basicos,Aco,USIM3.SA,9.300950,9.518752,9.300950,9.518752,55000
2019-02-27T00:00:00.000Z,Companhia Energética de Minas Gerais - CEMIG,Servicos Publicos,Utilidades - Eletricidade Regulada,CMIG4.SA,3.330564,3.396314,3.226272,3.253478,32201584
2019-02-27T00:00:00.000Z,Banco Santander (Brasil) S.A.,Financeiro,Bancos Regionais,SANB3.SA,14.078671,14.378090,14.078671,14.276289,5186
2019-02-28T00:00:00.000Z,Metalurgica Gerdau S.A.,Materiais Basicos,Aco,GOAU4.SA,3.010307,3.085774,3.010307,3.085774,8845645


### 5.3 Engenharia de Atributos

Para enriquecer a análise exploratória, criamos atributos derivados a partir das cotações:

- **`variacao_dia_pct`** — variação percentual entre abertura e fechamento do próprio dia;
- **`amplitude`** — diferença entre máxima e mínima do dia;
- **`retorno_diario_pct`** — variação percentual do fechamento em relação ao dia anterior (usa `lag` particionado por empresa);
- **`media_movel_20d`** — média móvel de 20 pregões do preço de fechamento;
- **`ano` / `mes` / `trimestre`** — colunas de apoio para agregações temporais na Seção 7.

In [0]:
janela_empresa = Window.partitionBy("cd_empresa").orderBy("data")
janela_movel_20d = janela_empresa.rowsBetween(-19, 0)

tb_acoes_final = (
    tb_acoes
    .withColumn("variacao_dia_pct", F.round((F.col("fechamento") - F.col("abertura")) / F.col("abertura") * 100, 2))
    .withColumn("amplitude", F.round(F.col("maxima") - F.col("minima"), 2))
    .withColumn(
        "retorno_diario_pct",
        F.round(
            (F.col("fechamento") - F.lag("fechamento").over(janela_empresa))
            / F.lag("fechamento").over(janela_empresa) * 100,
            2
        )
    )
    .withColumn("media_movel_20d", F.round(F.avg("fechamento").over(janela_movel_20d), 2))
    .withColumn("ano", F.year("data"))
    .withColumn("mes", F.month("data"))
    .withColumn("trimestre", F.quarter("data"))
)

print(f"Total de linhas após a engenharia de atributos: {tb_acoes_final.count()}")
display(tb_acoes_final.limit(10))

Total de linhas após a engenharia de atributos: 132902


data,empresa,setor,segmento,cd_empresa,Abertura,Maxima,Minima,Fechamento,Vol,variacao_dia_pct,amplitude,retorno_diario_pct,media_movel_20d,ano,mes,trimestre
2016-06-30T00:00:00.000Z,Banco Bradesco S.A.,Financeiro,Bancos Regionais,BBDC3.SA,7.885510,7.961417,7.710342,7.774570,2919210,-1.41,0.25,null,7.77,2016,6,2
2016-07-01T00:00:00.000Z,Banco Bradesco S.A.,Financeiro,Bancos Regionais,BBDC3.SA,7.958498,7.990613,7.841719,7.958498,2220957,0.00,0.15,2.37,7.87,2016,7,3
2016-07-04T00:00:00.000Z,Banco Bradesco S.A.,Financeiro,Bancos Regionais,BBDC3.SA,7.940168,8.036573,7.899271,7.998595,706369,0.74,0.14,0.50,7.91,2016,7,3
2016-07-05T00:00:00.000Z,Banco Bradesco S.A.,Financeiro,Bancos Regionais,BBDC3.SA,7.881743,7.946012,7.770732,7.887585,1395732,0.07,0.18,-1.39,7.90,2016,7,3
2016-07-06T00:00:00.000Z,Banco Bradesco S.A.,Financeiro,Bancos Regionais,BBDC3.SA,7.843767,7.910958,7.723993,7.799947,2961148,-0.56,0.19,-1.11,7.88,2016,7,3
2016-07-07T00:00:00.000Z,Banco Bradesco S.A.,Financeiro,Bancos Regionais,BBDC3.SA,7.875907,8.019052,7.811637,7.887592,2246274,0.15,0.21,1.12,7.88,2016,7,3
2016-07-08T00:00:00.000Z,Banco Bradesco S.A.,Financeiro,Bancos Regionais,BBDC3.SA,8.045334,8.132973,7.940166,7.983986,2107126,-0.76,0.19,1.22,7.90,2016,7,3
2016-07-11T00:00:00.000Z,Banco Bradesco S.A.,Financeiro,Bancos Regionais,BBDC3.SA,8.062861,8.217692,8.030727,8.068704,3515227,0.07,0.19,1.06,7.92,2016,7,3
2016-07-12T00:00:00.000Z,Banco Bradesco S.A.,Financeiro,Bancos Regionais,BBDC3.SA,8.074550,8.194324,8.016123,8.168033,3198472,1.16,0.18,1.23,7.95,2016,7,3
2016-07-13T00:00:00.000Z,Banco Bradesco S.A.,Financeiro,Bancos Regionais,BBDC3.SA,8.197243,8.223535,8.019042,8.033648,1978414,-2.00,0.20,-1.65,7.96,2016,7,3


## 6. Persistência da Camada Prata

Salvamos `tb_acoes_final` como a tabela curada da camada Prata. Diferente de uma abordagem com `CREATE TABLE` (DDL) seguido de um `INSERT` separado, o `saveAsTable` já cria a tabela **e** grava os dados em um único passo — evitando o risco de acabar com uma tabela vazia.

In [0]:
tb_acoes_final.write.format("delta").mode("overwrite").saveAsTable("workspace.prata.table_acoes_b3")

print("Tabela Prata salva com sucesso: prata.table_acoes_b3")

Tabela Prata salva com sucesso: prata.table_acoes_b3


## 7. Análise Exploratória de Dados (EDA)

Com a base tratada e enriquecida (`tb_acoes_final`), exploramos estatísticas gerais, o comportamento por empresa, por setor, o volume negociado e a evolução de preços no período.

### 7.1 Estatísticas Gerais

Estatísticas descritivas (contagem, média, desvio-padrão, mínimo e máximo) das principais colunas numéricas.

In [0]:
display(
    tb_acoes_final.describe(
        "abertura", "maxima", "minima", "fechamento", "vol", "retorno_diario_pct"
    )
)

summary,abertura,maxima,minima,fechamento,vol,retorno_diario_pct
count,132902,132902,132902,132902,132902,132846
mean,60.7219877840,62.1519051101,59.3720554579,60.8071871305,8967281.478397617,0.086141
stddev,506.31140122323427,519.2886492962559,494.3002187234018,507.322783798261,1.5200712321460446E7,2.9548771538266556
min,0.254591,0.257786,0.251197,0.254591,0,-76.65
max,12284.267578,12427.489123,11821.170233,12250.145527,630648029,200.75


### 7.2 Empresas

Quantidade de empresas/tickers analisados e volume de registros por empresa.

In [0]:
qtd_tickers = tb_acoes_final.select("cd_empresa").distinct().count()
print(f"Total de tickers analisados: {qtd_tickers}")

registros_por_empresa = (
    tb_acoes_final
    .groupBy("cd_empresa", "empresa")
    .count()
    .orderBy(F.desc("count"))
)
display(registros_por_empresa.limit(20))

Total de tickers analisados: 56


cd_empresa,empresa,count
AMER3.SA,Americanas S.A.,2489
BGIP3.SA,Banco do Estado de Sergipe S.A.,2489
UGPA3.SA,Ultrapar Participações S.A.,2489
YDUQ3.SA,Yduqs Participações S.A.,2489
PCAR3.SA,Companhia Brasileira De Distribuicao,2489
PRIO3.SA,Prio S.A.,2489
GOAU4.SA,Metalurgica Gerdau S.A.,2489
RENT3.SA,Localiza Rent a Car S.A.,2489
SANB3.SA,Banco Santander (Brasil) S.A.,2489
CMIG4.SA,Companhia Energética de Minas Gerais - CEMIG,2489


### 7.3 Setores

Comparação entre os setores: quantidade de empresas, quantidade de registros, preço médio de fechamento e retorno médio diário.

In [0]:
resumo_setores = (
    tb_acoes_final
    .groupBy("setor")
    .agg(
        F.countDistinct("cd_empresa").alias("qtd_empresas"),
        F.count("*").alias("qtd_registros"),
        F.round(F.avg("fechamento"), 2).alias("preco_medio_fechamento"),
        F.round(F.avg("retorno_diario_pct"), 3).alias("retorno_medio_diario_pct"),
    )
    .orderBy(F.desc("qtd_empresas"))
)
display(resumo_setores)

setor,qtd_empresas,qtd_registros,preco_medio_fechamento,retorno_medio_diario_pct
Financeiro,22,53903,14.88,0.092
Materiais Basicos,9,21246,12.26,0.100
Consumo Defensivo,6,13774,10.11,0.023
Servicos Publicos,5,11176,10.08,0.080
Industrial,4,8813,15.05,0.072
Consumo Cíclico,4,9956,643.08,0.062
Energia,4,9956,16.21,0.159
Tecnologia,2,4078,17.21,0.073


### 7.4 Volume

Identificamos as ações mais e menos líquidas (maior/menor volume médio diário) e o volume total negociado por setor.

In [0]:
volume_medio_empresa = (
    tb_acoes_final
    .groupBy("cd_empresa", "empresa", "setor")
    .agg(F.round(F.avg("vol"), 0).alias("volume_medio_diario"))
    .orderBy(F.desc("volume_medio_diario"))
)
display(volume_medio_empresa.limit(10))

cd_empresa,empresa,setor,volume_medio_diario
PETR4.SA,Petróleo Brasileiro S.A. - Petrobras,Energia,5.728594E7
BBDC4.SA,Banco Bradesco S.A.,Financeiro,3.5274383E7
COGN3.SA,Cogna Educação S.A.,Consumo Defensivo,3.1496984E7
ITUB4.SA,Itaú Unibanco Holding S.A.,Financeiro,3.0307772E7
ITSA4.SA,Itaúsa S.A.,Industrial,2.9746835E7
BBAS3.SA,Banco do Brasil S.A.,Financeiro,2.5446769E7
ABEV3.SA,Ambev S.A.,Consumo Defensivo,2.3877615E7
VALE3.SA,Vale S.A.,Materiais Basicos,2.2512586E7
RAIZ4.SA,Raízen S.A.,Servicos Publicos,1.8957029E7
USIM5.SA,Usinas Siderúrgicas de Minas Gerais S.A.,Materiais Basicos,1.6831714E7


In [0]:
volume_total_setor = (
    tb_acoes_final
    .groupBy("setor")
    .agg(F.sum("vol").alias("volume_total"))
    .orderBy(F.desc("volume_total"))
)
display(volume_total_setor)

setor,volume_total
Financeiro,257381281301
Energia,226556591037
Materiais Basicos,212155794755
Consumo Defensivo,193856302061
Industrial,102426267046
Servicos Publicos,101567274887
Consumo Cíclico,75115794587
Tecnologia,22710337368


### 7.5 Preços

Comparamos o preço de fechamento no início e no fim do período para identificar as maiores valorizações e desvalorizações, além das ações mais voláteis (maior desvio-padrão do retorno diário).

In [0]:
janela_periodo_completo = Window.partitionBy("cd_empresa").orderBy("data").rowsBetween(
    Window.unboundedPreceding, Window.unboundedFollowing
)

valorizacao_periodo = (
    tb_acoes_final
    .withColumn("preco_inicial", F.first("fechamento").over(janela_periodo_completo))
    .withColumn("preco_final", F.last("fechamento").over(janela_periodo_completo))
    .select("cd_empresa", "empresa", "setor", "preco_inicial", "preco_final")
    .distinct()
    .withColumn(
        "valorizacao_pct",
        F.round((F.col("preco_final") - F.col("preco_inicial")) / F.col("preco_inicial") * 100, 2)
    )
)

print("Maiores valorizações no período:")
display(valorizacao_periodo.orderBy(F.desc("valorizacao_pct")).limit(10))

Maiores valorizações no período:


cd_empresa,empresa,setor,preco_inicial,preco_final,valorizacao_pct
PRIO3.SA,Prio S.A.,Energia,0.279551,53.150002,18912.63
BMEB4.SA,Banco Mercantil do Brasil S.A.,Financeiro,1.338439,69.389999,5084.40
BMEB3.SA,Banco Mercantil do Brasil S.A.,Financeiro,2.186189,51.490002,2255.24
PETR4.SA,Petróleo Brasileiro S.A. - Petrobras,Energia,2.433882,38.070000,1464.17
PETR3.SA,Petróleo Brasileiro S.A. - Petrobras,Energia,3.176133,42.369999,1234.01
ENEV3.SA,Eneva S.A.,Servicos Publicos,2.252500,26.120001,1059.60
GOAU4.SA,Metalurgica Gerdau S.A.,Materiais Basicos,0.821533,9.510000,1057.59
GOAU3.SA,Metalurgica Gerdau S.A.,Materiais Basicos,0.818547,8.710000,964.08
VALE3.SA,Vale S.A.,Materiais Basicos,7.856224,78.180000,895.13
GGBR3.SA,Gerdau S.A.,Materiais Basicos,1.913168,18.209999,851.82


In [0]:
print("Maiores desvalorizações no período:")
display(valorizacao_periodo.orderBy(F.asc("valorizacao_pct")).limit(10))

Maiores desvalorizações no período:


cd_empresa,empresa,setor,preco_inicial,preco_final,valorizacao_pct
AMER3.SA,Americanas S.A.,Consumo Cíclico,872.295208,4.000000,-99.54
RAIZ4.SA,Raízen S.A.,Servicos Publicos,6.596122,0.420000,-93.63
COGN3.SA,Cogna Educação S.A.,Consumo Defensivo,10.619523,2.280000,-78.53
BMIN3.SA,Banco Mercantil de Investimentos S.A.,Financeiro,82.737862,21.490000,-74.03
VAMO3.SA,"Vamos Locação de Caminhões, Máquinas e Equipamentos S.A.",Industrial,7.094164,2.830000,-60.11
PCAR3.SA,Companhia Brasileira De Distribuicao,Consumo Cíclico,4.587934,2.290000,-50.09
BEEF3.SA,Minerva S.A.,Consumo Defensivo,6.782287,3.520000,-48.10
ASAI3.SA,Sendas Distribuidora S.A.,Consumo Defensivo,13.674992,8.580000,-37.26
ANIM3.SA,Ânima Holding S.A.,Consumo Defensivo,3.879446,2.690000,-30.66
YDUQ3.SA,Yduqs Participações S.A.,Consumo Defensivo,11.961917,8.800000,-26.43


In [0]:
volatilidade_empresa = (
    tb_acoes_final
    .groupBy("cd_empresa", "empresa", "setor")
    .agg(F.round(F.stddev("retorno_diario_pct"), 2).alias("volatilidade_pct"))
    .orderBy(F.desc("volatilidade_pct"))
)

print("Ações mais voláteis (maior desvio-padrão do retorno diário):")
display(volatilidade_empresa.limit(10))

Ações mais voláteis (maior desvio-padrão do retorno diário):


cd_empresa,empresa,setor,volatilidade_pct
AMER3.SA,Americanas S.A.,Consumo Cíclico,6.73
PCAR3.SA,Companhia Brasileira De Distribuicao,Consumo Cíclico,4.97
BMIN4.SA,Banco Mercantil de Investimentos S.A.,Financeiro,4.79
BRSR5.SA,Banco do Estado do Rio Grande do Sul S.A.,Financeiro,4.24
MGLU3.SA,Magazine Luiza S.A.,Consumo Cíclico,4.1
LWSA3.SA,Locaweb Serviços de Internet S.A.,Tecnologia,3.99
BRSR3.SA,Banco do Estado do Rio Grande do Sul S.A.,Financeiro,3.79
PRIO3.SA,Prio S.A.,Energia,3.72
BMIN3.SA,Banco Mercantil de Investimentos S.A.,Financeiro,3.66
ANIM3.SA,Ânima Holding S.A.,Consumo Defensivo,3.65


## 8. Conclusões

Este notebook concluiu a **Camada Prata** do projeto: leu as tabelas brutas da Bronze, aplicou limpeza, padronização e engenharia de atributos, persistiu a tabela curada `prata.acoes_b3`, e gerou a análise exploratória e os dois datasets finais (Power BI e Analytics).

### Estrutura entregue
- Tabela curada **`prata.acoes_b3`**, com 10 anos de histórico de mais de 50 ações da B3, setor/indústria traduzidos para português e atributos derivados (retorno diário, amplitude, variação do dia, média móvel de 20 dias);

### Próximos passos sugeridos
- Adicionar uma **Camada Ouro**: tabelas agregadas (por setor, por mês) prontas para consumo direto em dashboards, sem precisar reprocessar a granularidade diária;
- Adicionar indicadores técnicos adicionais (RSI, MACD, Bandas de Bollinger) na engenharia de atributos;
- Automatizar a atualização diária com um job agendado no Databricks (Bronze → Prata em sequência);
- Cruzar os dados de preço com indicadores macroeconômicos (Selic, IPCA, câmbio).

> ⚠️ **Nota de execução:** como este notebook depende de um cluster/sessão Spark e das tabelas geradas pela Camada Bronze, os resultados numéricos só estarão disponíveis após você executar `Finance_project_Bronze.ipynb` e, em seguida, este notebook completo no seu ambiente Databricks.